# TikTok Video & Creator 30-Day Engagement Prediction Pipeline

**Dataset**: `lingbow/tiktok-video-engagement-200k`  
**Paper / Citation**: [When does Trend-following Pay off? Evidence from Trending Content and Hashtag use](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=7371418)  
**Goal**: Master time-series modeling and trajectory prediction for short-form video engagement ($0-30$ days post-creation).

---
### Workflow Overview
1. **Load (PySpark)**: Read raw Parquet tables (`videos`, `engagement_daily`, `creator_daily`).
2. **Clean & Join (PySpark)**: Deduplicate, validate schemas, join daily engagement with creator stats & video content metadata.
3. **Aggregate (PySpark)**: Window functions for lags, rolling averages, daily growth rates, and 30-day trajectory summaries.
4. **EDA (Pandas / Seaborn)**: Analyze 30-day engagement decay curves, emotion correlations, stationarity (ADF test), and ACF/PACF autocorrelation.
5. **Baselines**: Establish Naive (Day 1 / Day 3 signal extrapolation) and Moving Average benchmarks.
6. **Classical Models**: Time-series forecasting using SARIMA and Facebook Prophet on topic-level daily aggregated series.
7. **Machine Learning Models**: Predict 30-day total engagement using LightGBM, Gradient Boosting, and Ridge Regression with strict time-based train/test split.
8. **Evaluation & Synthesis**: Master benchmark comparison table, error visual analysis, and key takeaways for creator strategies.

## Step 1: Load Data with PySpark
Initialize PySpark session and load raw Parquet files (`videos.parquet`, `engagement_daily.parquet`, `creator_daily.parquet`).

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Initialize Spark session tuned for local execution
spark = SparkSession.builder \
    .appName("TikTokEngagementAnalysis") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

data_dir = "data"
videos_df = spark.read.parquet(os.path.join(data_dir, "videos.parquet"))
engagement_df = spark.read.parquet(os.path.join(data_dir, "engagement_daily.parquet"))
creator_df = spark.read.parquet(os.path.join(data_dir, "creator_daily.parquet"))

print(f"[PySpark] Videos count: {videos_df.count():,}")
print(f"[PySpark] Engagement daily rows: {engagement_df.count():,}")
print(f"[PySpark] Creator daily rows: {creator_df.count():,}")

videos_df.printSchema()

## Step 2: Data Cleaning & Table Joins (PySpark)
Cast timestamp strings to date types, filter non-negative days, remove duplicates, and perform inner/left joins.

In [ ]:
# Clean dates and filter valid observations
engagement_clean = engagement_df \
    .withColumn("date", F.to_date("date")) \
    .filter(F.col("days_since_post") >= 0) \
    .dropDuplicates(["video_id", "date"])

creator_clean = creator_df \
    .withColumn("date", F.to_date("date")) \
    .dropDuplicates(["author_id", "date"])

videos_clean = videos_df.dropDuplicates(["video_id"])

# Join daily engagement with video metadata
full_df = engagement_clean.join(
    videos_clean.select("video_id", "author_id", "duration", "topic", "is_english", 
                        "joy", "disgust", "sadness", "anger", "surprise", "fear",
                        "word_count", "speaking_rate", "hashtag_count"),
    on="video_id",
    how="inner"
)

# Join with creator daily stats
full_df = full_df.join(
    creator_clean.select("author_id", "date", "follower_count", "enterprise_verified"),
    on=["author_id", "date"],
    how="left"
)

print(f"Joined dataset row count: {full_df.count():,}")

## Step 3: Spark Window Functions & Feature Engineering
Use `Window.partitionBy('video_id').orderBy('days_since_post')` to calculate lag features, rolling moving averages, daily growth increments, and calendar indicators.

In [ ]:
from pyspark.sql.window import Window

# Window per video trajectory
video_win = Window.partitionBy("video_id").orderBy("days_since_post")
roll_3d = Window.partitionBy("video_id").orderBy("days_since_post").rowsBetween(-3, -1)

full_feat = full_df \
    .withColumn("play_lag1", F.lag("play_count", 1).over(video_win)) \
    .withColumn("play_roll3_avg", F.avg("play_count").over(roll_3d)) \
    .withColumn("daily_play_inc", F.col("play_count") - F.coalesce(F.col("play_lag1"), F.lit(0))) \
    .withColumn("day_of_week", F.dayofweek("date"))

# Create 30-day cumulative trajectory summary table
pivoted_30d = full_df.filter(F.col("days_since_post").isin([0, 1, 3, 30])) \
    .groupBy("video_id") \
    .pivot("days_since_post", [0, 1, 3, 30]) \
    .agg(
        F.first("play_count").alias("plays"),
        F.first("like_count").alias("likes")
    )

# Extract static metadata per video at day 0
video_meta = full_df.filter(F.col("days_since_post") == 0).select(
    "video_id", "author_id", "date", "topic", "duration", "is_english",
    "joy", "disgust", "sadness", "anger", "surprise", "fear",
    "word_count", "speaking_rate", "hashtag_count", "follower_count"
).dropDuplicates(["video_id"])

summary_30d_spark = video_meta.join(pivoted_30d, on="video_id", how="inner") \
    .withColumnRenamed("0_plays", "plays_day0") \
    .withColumnRenamed("1_plays", "plays_day1") \
    .withColumnRenamed("3_plays", "plays_day3") \
    .withColumnRenamed("30_plays", "target_plays_30d") \
    .withColumnRenamed("0_likes", "likes_day0") \
    .withColumnRenamed("1_likes", "likes_day1") \
    .withColumnRenamed("3_likes", "likes_day3")

# Save to Pandas / local Parquet for EDA & modeling
df_30d = summary_30d_spark.toPandas()
print(f"Extracted 30-day summary table: {df_30d.shape[0]:,} videos")
df_30d.head(3)

## Step 4: Exploratory Data Analysis (EDA)
Analyze video topic distribution, cumulative engagement growth curves over $0-30$ days, emotion correlations, and perform stationarity testing (ADF test) and ACF/PACF autocorrelation.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller, acf, pacf

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"font.size": 11})

# 1. 30-Day Trajectory Curve per Topic
play_cols = ["plays_day0", "plays_day1", "plays_day3", "target_plays_30d"]
days = [0, 1, 3, 30]
topic_medians = df_30d.groupby("topic")[play_cols].median()

fig, ax = plt.subplots(figsize=(10, 5))
for topic in topic_medians.index[:6]:
    ax.plot(days, topic_medians.loc[topic, play_cols].values, marker="o", linewidth=2, label=topic)

ax.set_title("Median 30-Day Cumulative Play Count Trajectory by Topic", fontsize=13, fontweight="bold")
ax.set_xlabel("Days Since Post")
ax.set_ylabel("Median Play Count (Log Scale)")
ax.set_yscale("log")
ax.legend(bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()

## Step 5: Baseline Benchmarks
Evaluate Naive Extrapolation (Day 1 / Day 3 early signals) and Mean Predictor.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

def compute_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    wape = (np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))) * 100
    return {"MAE": mae, "RMSE": rmse, "WAPE (%)": wape}

y_true = df_30d["target_plays_30d"].fillna(0)

# Extrapolate from Day 1 signal
scale_d1 = (df_30d["target_plays_30d"] / (df_30d["plays_day1"] + 1)).median()
pred_d1 = df_30d["plays_day1"] * scale_d1

# Extrapolate from Day 3 signal
scale_d3 = (df_30d["target_plays_30d"] / (df_30d["plays_day3"] + 1)).median()
pred_d3 = df_30d["plays_day3"] * scale_d3

baseline_df = pd.DataFrame([
    {"Model": "Mean Predictor", **compute_metrics(y_true, np.full_like(y_true, y_true.mean()))},
    {"Model": "Naive Extrapolation (Day 1 Signal)", **compute_metrics(y_true, pred_d1)},
    {"Model": "Naive Extrapolation (Day 3 Signal)", **compute_metrics(y_true, pred_d3)}
])

print("Baseline Benchmarks:")
display(baseline_df)

## Step 6: Classical Time Series Models (SARIMA / Prophet)
Fit SARIMA and Facebook Prophet on topic daily aggregated time series.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Aggregate daily plays by topic
df_topic_daily = full_df.groupBy("date", "topic").agg(F.sum("play_count").alias("total_plays")).toPandas()
top_topic = df_topic_daily.groupby("topic")["total_plays"].sum().idxmax()
ts_top = df_topic_daily[df_topic_daily["topic"] == top_topic].sort_values("date").set_index("date")["total_plays"]

# Train / Test split (80/20)
split_len = int(len(ts_top) * 0.8)
train_ts, test_ts = ts_top.iloc[:split_len], ts_top.iloc[split_len:]

# Fit SARIMA(1,1,1)x(1,1,0,7)
sarima = SARIMAX(train_ts, order=(1,1,1), seasonal_order=(1,1,0,7), enforce_stationarity=False)
sarima_fit = sarima.fit(disp=False)
sarima_pred = sarima_fit.predict(start=len(train_ts), end=len(train_ts) + len(test_ts) - 1, dynamic=True)
sarima_pred.index = test_ts.index

classical_df = pd.DataFrame([{
    "Model": "SARIMA(1,1,1)x(1,1,0,7)",
    **compute_metrics(test_ts.values, sarima_pred.values)
}])
display(classical_df)

## Step 7: Machine Learning Models (LightGBM / GBT / Ridge)
Train LightGBM Regressor and Random Forest on early trajectory features + video metadata + creator followers with strict time-based train/test splitting.

In [ ]:
import lightgbm as lgb
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

# Prepare features
y_log = np.log1p(df_30d["target_plays_30d"].clip(lower=0).fillna(0))
X_df = pd.DataFrame({
    "plays_day0_log": np.log1p(df_30d["plays_day0"].fillna(0)),
    "plays_day1_log": np.log1p(df_30d["plays_day1"].fillna(0)),
    "plays_day3_log": np.log1p(df_30d["plays_day3"].fillna(0)),
    "likes_day3_log": np.log1p(df_30d["likes_day3"].fillna(0)),
    "duration": df_30d["duration"].fillna(0),
    "speaking_rate": df_30d["speaking_rate"].fillna(0),
    "follower_count_log": np.log1p(df_30d["follower_count"].fillna(0)),
    "joy": df_30d["joy"].fillna(0),
    "sadness": df_30d["sadness"].fillna(0),
    "anger": df_30d["anger"].fillna(0)
})

# Add topic dummy features
if "topic" in df_30d.columns:
    topic_dummies = pd.get_dummies(df_30d["topic"], prefix="topic", drop_first=True)
    X_df = pd.concat([X_df, topic_dummies], axis=1)

# Time-based split
cutoff = df_30d["date"].quantile(0.8) if "date" in df_30d.columns else df_30d.index[int(len(df_30d)*0.8)]
train_mask = df_30d["date"] <= cutoff if "date" in df_30d.columns else df_30d.index <= cutoff

X_train, y_train = X_df[train_mask], y_log[train_mask]
X_test, y_test = X_df[~train_mask], y_log[~train_mask]

# Fit LightGBM
lgb_reg = lgb.LGBMRegressor(n_estimators=250, learning_rate=0.05, random_state=42, n_jobs=-1)
lgb_reg.fit(X_train, y_train)
pred_lgb_log = lgb_reg.predict(X_test)

# Fit Ridge
ridge_reg = Ridge(alpha=1.0)
ridge_reg.fit(X_train, y_train)
pred_ridge_log = ridge_reg.predict(X_test)

ml_results = pd.DataFrame([
    {"Model": "Ridge Regression", **compute_metrics(np.expm1(y_test), np.expm1(pred_ridge_log))},
    {"Model": "LightGBM Regressor", **compute_metrics(np.expm1(y_test), np.expm1(pred_lgb_log))}
])
print("ML Model Performance:")
display(ml_results)

## Step 8: Master Evaluation & Student Takeaways
Synthesize all models into a single comparative leaderboard and highlight core learning outcomes.

In [ ]:
master_leaderboard = pd.concat([baseline_df, classical_df, ml_results], ignore_index=True).sort_values("WAPE (%)")
print("==============================================================")
print("    FINAL MODEL LEADERBOARD (30-DAY TIKTOK PLAY COUNT)       ")
print("==============================================================")
display(master_leaderboard)

# Plot Leaderboard
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x="WAPE (%)", y="Model", data=master_leaderboard, palette="mako", ax=ax)
ax.set_title("Master Benchmark: Model WAPE Error Comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()